# Inherited Globe — Data Pipeline

Builds `sites.geojson` for the globe at `index.html`, from two official UNESCO exports:
the World Heritage List in `data/whc001.csv` and the Global Geoparks list in
`data/eg0001.csv`.

Five channels:

| # | Channel | What it gives |
|---|---|---|
| 1 | UNESCO World Heritage List export (CSV) | names, category, danger status, region, states, criteria, area, coordinates, components, **all photos** |
| 2 | UNESCO Global Geoparks export (CSV) | names, designation year, country, area, population, coordinates, introduction, **its one photo** — `Main Image` is the export's only image column, so there is no gallery to take — video and website links |
| 3 | Wikidata (SPARQL) | World Heritage site ID → article, via `P757`; geopark → article, via the designation's own items |
| 4 | Wikimedia Pageviews (REST) | 12-month human traffic per article → popularity |
| 5 | Wikipedia lead images | a photo for the 56 sites the exports leave without one |

Photos come from the exports, with Wikipedia as a fallback. 56 sites (12 World Heritage
properties, 44 geoparks) have an empty image column, and section 7.1 gives those the lead
image of their Wikipedia article, credited to its Commons author and licence. The
`image_source` field says which of the two a photo came from.

UNESCO publishes originals rather than web assets, so the globe requests each photo
through a resizing proxy at the width the card shows (section 7.2). Seven photos are too
large for that proxy to ingest; those seven the pipeline downloads and shrinks itself,
and they are the only images it ever fetches.

Only channels 1–2 are mandatory. Channels 3–5 hit public APIs and take roughly 30–50 min
for both lists; each stage has a checkpoint so a run can be resumed.

Run the cells top to bottom. The heavy cells are gated by the `RUN_*` flags in the
configuration cell.

---

In [1]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# pip install pandas requests tqdm pillow matplotlib
# Optional: ipywidgets, for graphical progress bars instead of text ones.
import importlib
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from scripts import pipeline_helpers as ph

importlib.reload(ph)
pd.set_option("display.max_colwidth", 90)
print("helpers loaded from", ph.__file__)

helpers loaded from /Users/thomasdemareuil/Documents/PERSO/inherited-globe/scripts/pipeline_helpers.py


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
### Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
WHC_CSV        = "data/whc001.csv"         # official UNESCO World Heritage List export
GEOPARK_CSV    = "data/eg0001.csv"         # official UNESCO Global Geoparks export
OUTPUT_PATH    = "sites.geojson"           # what index.html fetches
CACHE_DIR      = Path("data/cache")
CKPT_DIR       = CACHE_DIR / "checkpoints"
WIKIDATA_CACHE = CACHE_DIR / "wikidata" / "wikidata_map.json"
NAME_CACHE     = CACHE_DIR / "wikidata" / "name_fallback.json"
GEOPARK_WIKIDATA_CACHE = CACHE_DIR / "wikidata" / "geopark_map.json"
GEOPARK_NAME_CACHE     = CACHE_DIR / "wikidata" / "geopark_name_fallback.json"
SECRETS_DIR    = Path("data/secrets")      # git-ignored
IMAGE_DIR      = Path("images")            # committed: photos shrunk for the globe
PROXY_CACHE    = CACHE_DIR / "images" / "proxy_check.json"
THUMBNAIL_CACHE = CACHE_DIR / "images" / "wikipedia_thumbnails.json"

for folder in (CKPT_DIR, WIKIDATA_CACHE.parent, CACHE_DIR / "pageviews",
               PROXY_CACHE.parent):
    folder.mkdir(parents=True, exist_ok=True)

# ── Which network stages to run ───────────────────────────────────────────────
# Nothing here touches images; all three fetch Wikipedia metadata only.
RUN_WIKIDATA_BATCH = True   # channel 2, pass 1 — fast (a few SPARQL batches)
RUN_NAME_FALLBACK  = True   # channel 2, pass 2 — slow, one lookup per unresolved site
RUN_PAGEVIEWS      = True   # channel 4 — ~1 request/second per resolved article

# Geoparks (section 6). Same two-pass shape, but the first pass matches on the name:
# Wikidata has no UNESCO Global Geopark identifier property to join on.
RUN_GEOPARK_DESIGNATION   = True   # one SPARQL query, then one batch of sitelinks
RUN_GEOPARK_NAME_FALLBACK = True   # slow, one lookup per geopark left unresolved
RUN_GEOPARK_PAGEVIEWS     = True   # ~1 request/second per resolved article

# Cross-check both lists against Wikipedia's own index pages (sections 4.1 and 6.2).
RUN_ARTICLE_CROSSCHECK = True   # 7 page fetches, a few seconds

# Photos (section 7). The thumbnail lookup covers only the sites whose export ships no
# photo; the check is one HEAD per main photo through the proxy; the reduction downloads
# only the photos that check refuses.
RUN_WIKIPEDIA_THUMBNAILS = True   # 2 requests per photo-less site (56 of them)
RUN_PROXY_CHECK          = True   # ~1,460 HEAD requests, 8 at a time, ~3 min (cached)
RUN_LOCAL_REDUCTION      = True   # downloads and shrinks what the proxy cannot ingest

# ── Label points ──────────────────────────────────────────────────────────────
# Serial properties publish one coordinate per component (up to 758 of them). Components
# are clustered by distance, and each kept cluster is reduced to its centroid. The largest
# cluster always gets a point; secondary clusters only if they hold enough components.
COMPONENT_CLUSTER_BUFFER_KM = 250      # single-linkage distance between components
MAX_LABEL_POINTS_PER_SITE   = 1        # raise to 2 to also label big secondary clusters
SECONDARY_CLUSTER_MIN_SHARE = 0.25     # a secondary cluster needs >= 25% of components
JITTER_DUPLICATE_POINTS     = True     # nudge apart sites sharing exact coordinates

# ── Manual coordinates ────────────────────────────────────────────────────────
# For the properties the export locates neither by `Coordonnées` nor by `Components`.
# Without an entry here such a property cannot be placed and is dropped from the globe.
#
# 1567 — "Funerary and memory sites of the First World War (Western Front)": a
# transnational serial property of 139 components strung along the former Western Front,
# from the Belgian coast to the Vosges. The export ships no coordinate and no component
# list for it. Placed on the Thiepval Memorial to the Missing of the Somme: the largest
# Commonwealth memorial on the Western Front, in the Somme/Artois sector where the
# property's funerary and memorial components are most densely concentrated.
MANUAL_COORDINATES = {
    1567: (50.0508, 2.6866),   # Thiepval Memorial, Somme, France
}

# ── Popularity ────────────────────────────────────────────────────────────────
# Last 12 completed months. Wikimedia's monthly endpoint expects YYYYMMDD.
PAGEVIEW_START = "20250601"
PAGEVIEW_END   = "20260531"
MIN_POPULARITY = 1   # floor on the view count, so no label is ever sorted out entirely

# ── Popup content ─────────────────────────────────────────────────────────────
MAX_EXTRA_IMAGES      = None  # None = keep every secondary photo the export lists
DESCRIPTION_MAX_CHARS = None  # None = export the full text; the popup flip face scrolls
SHORT_LABEL_MAX_CHARS = 42    # on-globe label length before the full name is shortened

# ── Credits / versioning, shown in the globe's Method panel ───────────────────
WHC_EXPORT_DATE     = "2026-09-16"   # the day data/whc001.csv was downloaded
GEOPARK_EXPORT_DATE = "2026-09-17"   # the day data/eg0001.csv was downloaded

# Optional: raises the Wikimedia rate limit from 500 to 5,000 req/hour.
# Create one at https://api.wikimedia.org/ and store it in the git-ignored file below.
WIKIMEDIA_TOKEN = ph.read_local_secret(SECRETS_DIR / "wikimedia_token.txt")

ph.configure(
    WIKIMEDIA_TOKEN=WIKIMEDIA_TOKEN,
    USER_AGENT="InheritedGlobe/1.0 (https://github.com/tdemareuil/inherited-globe)",
)
ph.set_pageview_window(PAGEVIEW_START, PAGEVIEW_END)

print("Wikimedia token:", "set" if WIKIMEDIA_TOKEN else "not set (500 req/hour)")
print("Pageview window:", PAGEVIEW_START, "→", PAGEVIEW_END)
print("Manual coordinates:", len(MANUAL_COORDINATES), "site(s)")

Wikimedia token: set
Pageview window: 20250601 → 20260531
Manual coordinates: 1 site(s)


---
## 1. Load the UNESCO export

The export is the file published at
[data.unesco.org](https://data.unesco.org/explore/assets/whc001/export/), one row per
inscribed property, in six UN languages. We keep the English fields plus the structured
ones, and drop the other five language blocks.

In [3]:
raw = pd.read_csv(WHC_CSV, encoding="utf-8-sig")
print(f"{len(raw):,} rows × {raw.shape[1]} columns")
raw[["Name EN", "Catégorie", "Danger", "Region", "States Names", "Date inscribed"]].head()

1,273 rows × 54 columns


,Name EN,Catégorie,Danger,Region,States Names,Date inscribed
0,The Bony Fish Fossils of the Western Limfjord – Evolution and Climate Adaptation in th...,Natural,False,Europe and North America,Denmark,2026
1,Aalto Works,Cultural,False,Europe and North America,Finland,2026
2,"Beaches of the D-Day Landings, Normandy, 1944",Cultural,False,Europe and North America,France,2026
3,Ancient Buddhist Site of Sarnath,Cultural,False,Asia and the Pacific,India,2026
4,Alamūt Castle and Related Fortifications,Cultural,False,Asia and the Pacific,Iran (Islamic Republic of),2026


In [4]:
# Column presence check — the export's schema has changed across versions.
EXPECTED = [
    "Name EN", "Short Description EN", "Date inscribed", "Danger", "Danger list",
    "Area hectares", "Criteria", "Catégorie", "States Names", "ISO Codes", "Region",
    "Transboundary", "Main Image", "Main Image Author", "Main Image Copyright",
    "Main Image Caption EN", "Images", "ID", "Coordonnées", "Components",
    "Components Count",
]
missing = [c for c in EXPECTED if c not in raw.columns]
print("missing columns:", missing or "none")

missing columns: none


In [5]:
def to_bool(value):
    """The export writes booleans as True/False, Y/N or 1/0 depending on the column."""
    text = ph.clean_str(value).lower()
    return text in {"true", "y", "yes", "1"}


sites = pd.DataFrame({
    "site_id":           raw["ID"].astype("Int64"),
    "label":             raw["Name EN"].map(ph.strip_tags),
    "short_description": raw["Short Description EN"].map(ph.description_text),
    "category":          raw["Catégorie"].map(lambda v: ph.clean_str(v).title()),
    "in_danger":         raw["Danger"].map(to_bool),
    "danger_list":       raw["Danger list"].map(ph.clean_str),
    "region":            raw["Region"].map(ph.clean_str),
    "states":            raw["States Names"].map(ph.clean_str),
    "iso_codes":         raw["ISO Codes"].map(lambda v: ph.clean_str(v).upper()),
    "date_inscribed":    pd.to_numeric(raw["Date inscribed"], errors="coerce").astype("Int64"),
    "criteria":          raw["Criteria"].map(ph.clean_str),
    "area_hectares":     pd.to_numeric(raw["Area hectares"], errors="coerce"),
    "transboundary":     raw["Transboundary"].map(to_bool),
    "component_count":   pd.to_numeric(raw["Components Count"], errors="coerce").fillna(0).astype(int),
    "_coordinates":      raw["Coordonnées"],
    "_components":       raw["Components"],
    "image_author":      raw["Main Image Author"].map(ph.strip_tags),
    "image_copyright":   raw["Main Image Copyright"].map(ph.strip_tags),
    "image_caption":     raw["Main Image Caption EN"].map(ph.strip_tags),
})

sites["short_label"] = sites["label"].map(lambda n: ph.short_label(n, SHORT_LABEL_MAX_CHARS))
sites["short_description"] = sites["short_description"].map(
    lambda t: ph.truncate_text(t, DESCRIPTION_MAX_CHARS))
sites["color_key"] = [
    ph.category_color_key(cat, danger)
    for cat, danger in zip(sites["category"], sites["in_danger"])
]
sites["unesco_url"] = sites["site_id"].map(ph.unesco_site_url)
sites["dataset"] = "whc"

print(sites["category"].value_counts().to_string())
print()
print("in danger:", int(sites["in_danger"].sum()))
print()
print(sites["region"].value_counts().to_string())
print()
unknown_regions = set(sites["region"]) - set(ph.UNESCO_REGIONS)
print("regions not in the index.html filter:", unknown_regions or "none")

category
Cultural    991
Natural     240
Mixed        42

in danger: 58

region
Europe and North America           588
Asia and the Pacific               313
Latin America and the Caribbean    155
Africa                             115
Arab States                        102

regions not in the index.html filter: none


In [6]:
# Sanity: duplicate site IDs, empty names, unexpected categories
print("duplicate site IDs:", int(sites["site_id"].duplicated().sum()))
print("empty names:", int((sites["label"] == "").sum()))
print("unexpected categories:", set(sites["category"]) - set(ph.CATEGORIES) or "none")
print("date range:", int(sites["date_inscribed"].min()), "→", int(sites["date_inscribed"].max()))
sites[sites["short_label"].notna()][["label", "short_label"]].head(10)

duplicate site IDs: 0
empty names: 0
unexpected categories: none
date range: 1978 → 2026


,label,short_label
0,The Bony Fish Fossils of the Western Limfjord – Evolution and Climate Adaptation in th...,The Bony Fish Fossils of the Western…
2,"Beaches of the D-Day Landings, Normandy, 1944","Beaches of the D-Day Landings, Normandy"
6,The Cemetery Complexes of the Xiongnu Nobility,The Cemetery Complexes of the Xiongnu…
9,"Wat Phra Mahathat Woramahawihan, Nakhon Si Thammarat",Wat Phra Mahathat Woramahawihan
10,Tashkent Modernist Architecture. Modernity and Tradition in Central Asia,Tashkent Modernist Architecture.…
11,Jingdezhen Handicraft Porcelain Industry Sites,Jingdezhen Handicraft Porcelain Industry…
12,The system of Italian-style condominio theatres of the 18th and 19th centuries in Cent...,The system of Italian-style condominio…
17,The Medinas of the Historic Sultanates of the Comoros,The Medinas of the Historic Sultanates of…
21,Rock Mosques and Associated Sacred Sites of Mangystau,Rock Mosques
22,The Roças of Sao Tome and Principe: Colonial Agricultural System and Forced Migration,The Roças of Sao Tome and Principe


---
## 2. Label points

`Coordonnées` and `Components` are independent columns, and all four combinations occur in
the export. The placement rule covers each of them:

| `Coordonnées` | `Components` | Placement | `point_source` |
|---|---|---|---|
| yes | yes | the published point, if it falls inside the largest component cluster; otherwise that cluster's centroid | `site_coordinates` / `component_centroid` |
| yes | no | the published point | `site_coordinates` |
| **no** | yes | **the centroid of the largest component cluster** | `component_centroid` |
| no | no | `MANUAL_COORDINATES`, or dropped | `manual` |

The centroid is a spherical mean, not a planar average, so a cluster straddling the
antimeridian or sitting at high latitude still lands in the right place.

Clustering exists so that a property split across a country — the 758 components of the
Frontiers of the Roman Empire, say — does not become 758 labels. With
`MAX_LABEL_POINTS_PER_SITE = 1` each property gets exactly one point, on its largest
cluster. Raise it to 2 and set `SHOW_MAIN_POINT_ONLY = false` in `index.html` to let big
secondary clusters carry a second label.

In [7]:
# What the export actually provides, before any placement.
coord_present = sites["_coordinates"].map(lambda v: ph.parse_coordinates(v) != (None, None))
comps_present = sites["_components"].map(lambda v: len(ph.parse_components(v)) > 0)

print(pd.crosstab(coord_present, comps_present,
                  rownames=["has Coordonnées"], colnames=["has Components"],
                  margins=True).to_string())
print()
needs_centroid = (~coord_present) & comps_present
needs_manual = (~coord_present) & (~comps_present)
print(f"{int(needs_centroid.sum())} site(s) will be placed on a computed centroid "
      f"(tabled, with the result, two cells down)")
print(f"{int(needs_manual.sum())} site(s) have neither, and need MANUAL_COORDINATES:")
for row in sites[needs_manual][["site_id", "label"]].itertuples(index=False):
    covered = "covered" if int(row[0]) in MANUAL_COORDINATES else "NOT COVERED — will be dropped"
    print(f"  {row[0]:5d}  {row[1][:70]}  → {covered}")

has Components   False  True   All
has Coordonnées                   
False                1    28    29
True                 1  1243  1244
All                  2  1271  1273

28 site(s) will be placed on a computed centroid (tabled, with the result, two cells down)
1 site(s) have neither, and need MANUAL_COORDINATES:
   1567  Funerary and memory sites of the First World War (Western Front)  → covered


In [8]:
records = []
no_location = []
centroid_ids = set(sites.loc[needs_centroid, "site_id"].astype(int))

for row in sites.to_dict("records"):
    site_id = int(row["site_id"])
    lat, lon = ph.parse_coordinates(row.pop("_coordinates"))
    components = ph.parse_components(row.pop("_components"))

    points = ph.site_label_points(
        lat, lon, components,
        buffer_km=COMPONENT_CLUSTER_BUFFER_KM,
        max_points=MAX_LABEL_POINTS_PER_SITE,
        secondary_min_share=SECONDARY_CLUSTER_MIN_SHARE,
    )

    # Neither a published coordinate nor a component: fall back to the hard-coded pick.
    if not points and site_id in MANUAL_COORDINATES:
        manual_lat, manual_lon = MANUAL_COORDINATES[site_id]
        points = [{
            "lat": manual_lat,
            "lon": manual_lon,
            "point_source": "manual",
            "cluster_component_count": 0,
            "cluster_share": 1.0,
        }]

    if not points:
        no_location.append((site_id, row["label"]))
        continue

    for rank, point in enumerate(points, start=1):
        record = dict(row)
        record.update({
            "lat": point["lat"],
            "lon": point["lon"],
            "point_source": point["point_source"],
            "cluster_component_count": point["cluster_component_count"],
            "cluster_share": point["cluster_share"],
            "label_rank": rank,
            "label_count": len(points),
        })
        records.append(record)

df = pd.DataFrame(records)
print(f"{df['site_id'].nunique():,} sites → {len(df):,} label points")
print(f"sites with more than one label point: {df.loc[df['label_count'] > 1, 'site_id'].nunique()}")
print()
print(df["point_source"].value_counts().to_string())
print()
if no_location:
    print(f"{len(no_location)} site(s) still unplaced — add them to MANUAL_COORDINATES:")
    for site_id, label in no_location:
        print(f"  {site_id}  {label}")
else:
    print("every site in the export is placed ✓")

1,273 sites → 1,273 label points
sites with more than one label point: 0

point_source
site_coordinates      1218
component_centroid      54
manual                   1

every site in the export is placed ✓


In [9]:
# Verify the centroid path: every site without published coordinates must now have a
# computed point, and it must sit inside the bounding box of its own components.
placed = df.drop_duplicates("site_id")
placed = placed.set_index(placed["site_id"].astype(int))
checks = []
for site_id in sorted(centroid_ids):
    components = ph.parse_components(raw.loc[raw["ID"] == site_id, "Components"].iloc[0])
    row = placed.loc[site_id]
    lats = [c["lat"] for c in components]
    lons = [c["lon"] for c in components]
    checks.append({
        "site_id": site_id,
        "label": row["label"][:46],
        "components": len(components),
        "in_cluster": row["cluster_component_count"],
        "lat": round(row["lat"], 4),
        "lon": round(row["lon"], 4),
        "source": row["point_source"],
        "within_bbox": bool(min(lats) - 0.01 <= row["lat"] <= max(lats) + 0.01
                            and min(lons) - 0.01 <= row["lon"] <= max(lons) + 0.01),
    })

checks = pd.DataFrame(checks)
print(f"{len(checks)} centroid placements — all from components: "
      f"{bool((checks['source'] == 'component_centroid').all())}")
print(f"all inside their own component bounding box: {bool(checks['within_bbox'].all())}")
checks

28 centroid placements — all from components: True
all inside their own component bounding box: True


,site_id,label,components,in_cluster,lat,lon,source,within_bbox
0,136,Garamba National Park,1,1,4.1667,29.4997,component_centroid,True
1,927,Ancient Buddhist Site of Sarnath,2,2,25.3775,83.0240,component_centroid,True
2,1239,Berlin Modernism Housing Estates,7,7,52.5035,13.3833,component_centroid,True
3,1581,"Beaches of the D-Day Landings, Normandy, 1944",6,6,49.3627,-0.7224,component_centroid,True
4,1591,"Getbol, Korean Tidal Flats",6,6,35.5639,126.6304,component_centroid,True
5,1715,Gdynia Modernist City Centre,1,1,54.5194,18.5425,component_centroid,True
6,1719,The wider area of Mount Olympus,2,2,40.1383,22.4412,component_centroid,True
7,1724,Wadi Wurayah,1,1,25.4823,56.2658,component_centroid,True
8,1742,Aqaba Marine Reserve,1,1,29.4297,34.9756,component_centroid,True
9,1750,The Roças of Sao Tome and Principe: Colonial A,6,6,0.7218,6.8897,component_centroid,True


In [10]:
# Sites whose published coordinate fell outside their largest cluster, and were therefore
# moved onto that cluster's centroid instead.
moved = df[(df["point_source"] == "component_centroid") & (~df["site_id"].astype(int).isin(centroid_ids))]
print(f"{moved['site_id'].nunique()} site(s) re-centred onto a component cluster")
moved[["site_id", "label", "component_count", "cluster_component_count", "lat", "lon"]].head(15)

26 site(s) re-centred onto a component cluster


,site_id,label,component_count,cluster_component_count,lat,lon
63,1723,The Emergence of Modern Human Behaviour: The Pleistocene Occupation Sites of South Africa,3,1,-29.522000,31.086000
99,1693,Cold Winter Deserts of Turan,14,6,45.238328,58.761312
129,1654,Petroglyphs of Lake Onega and the White Sea,2,1,64.491422,34.670603
165,1606,Migratory Bird Sanctuaries along the Coast of Yellow Sea-Bohai Gulf of China,12,9,38.613804,120.382317
175,1603,French Austral Lands and Seas,3,1,-46.255139,50.913167
182,1496,The 20th-Century Architecture of Frank Lloyd Wright,8,4,42.474322,-88.715658
231,1506,The Persian Qanat,11,6,32.952781,52.342530
252,1468,Moravian Church Settlements,4,1,51.015556,14.744167
286,1459,"Qhapaq Ñan, Andean Road System",137,35,-24.697590,-68.637355
291,1444,Pyu Ancient Cities,3,2,19.401117,95.334557


In [11]:
if JITTER_DUPLICATE_POINTS:
    df = ph.jitter_duplicate_points(df)

duplicates = df.groupby([df["lat"].round(6), df["lon"].round(6)]).size()
print("coordinate pairs still shared by 2+ label points:", int((duplicates > 1).sum()))

coordinate pairs still shared by 2+ label points: 0


### Checkpoint 1 — after parsing & placement

In [12]:
CKPT1 = CKPT_DIR / "ckpt_1_label_points.csv"
# Resume a later run from here by uncommenting: df = pd.read_csv(CKPT1)
df.to_csv(CKPT1, index=False)
print("saved", CKPT1, df.shape)

saved data/cache/checkpoints/ckpt_1_label_points.csv (1273, 28)


---
## 3. Photos — straight from the export

No network calls in this section. Each property's photos are exactly the URLs the export
lists for it: `Main Image` becomes the popup's primary photo, and every URL in `Images`
becomes a slide behind it. `Main Image Author` and `Main Image Copyright` become the
credit line.

The dozen properties with no photo in the export fall back to Wikipedia in section 7.1,
once their article is known.

`MAX_EXTRA_IMAGES = None` keeps every secondary photo (25,497 URLs across the List, about
20 per property, up to 145 for one). Set it to an integer to cap the slideshow and shrink
the exported file.

> **Hotlinking caveat.** UNESCO serves these photos from `whc.unesco.org/document/<id>`
> behind a bot challenge. Ordinary browsers load them; scripted requests generally get a
> 403, so image reachability cannot be verified here — it is verified by opening the globe.
> The popup sends `referrerpolicy="no-referrer"`, hides any image that fails to load, and
> steps to the next slide, so a blocked photo degrades quietly.

In [13]:
main_images = raw.set_index(raw["ID"].astype(str))["Main Image"]
gallery     = raw.set_index(raw["ID"].astype(str))["Images"]
keys        = df["site_id"].astype(str)


def primary_image(site_key):
    urls = ph.parse_url_list(main_images.get(site_key))
    return urls[0] if urls else None


def secondary_images(site_key):
    """Every gallery URL except the primary, in export order."""
    primary = primary_image(site_key)
    urls = [u for u in ph.parse_url_list(gallery.get(site_key)) if u != primary]
    return urls if MAX_EXTRA_IMAGES is None else urls[:MAX_EXTRA_IMAGES]


df["image_url"]        = keys.map(primary_image)
df["extra_image_urls"] = keys.map(lambda k: ", ".join(secondary_images(k)))
df["image_count"] = [
    (1 if primary else 0) + (len(extra.split(", ")) if extra else 0)
    for primary, extra in zip(df["image_url"], df["extra_image_urls"])
]
df["image_source"] = df["image_url"].map(lambda u: "UNESCO" if u else None)

# Author, copyright and caption describe the Main Image only — the gallery photos carry
# no per-photo metadata in the export — so they are cleared when there is no main photo.
no_primary = df["image_url"].isna()
for column in ("image_author", "image_copyright", "image_caption"):
    df.loc[no_primary, column] = None
    df[column] = df[column].replace("", None)

unique = df.drop_duplicates("site_id")
print("sites with a primary photo:", int(unique["image_url"].notna().sum()), "/", len(unique))
print("sites with secondary photos:", int((unique["extra_image_urls"] != "").sum()))
print("total photo URLs exported:", int(unique["image_count"].sum()))
print()
for column in ("image_author", "image_copyright", "image_caption"):
    print(f"{column:16s} present for {int(unique[column].notna().sum()):5d} sites")
print()
print(unique["image_count"].describe().round(1).to_string())

sites with a primary photo: 1261 / 1273
sites with secondary photos: 1214
total photo URLs exported: 25509

image_author     present for  1196 sites
image_copyright  present for  1216 sites
image_caption    present for  1261 sites

count    1273.0
mean       20.0
std        14.9
min         1.0
25%        11.0
50%        16.0
75%        26.0
max       145.0


In [14]:
# The export lists no photo for these properties; section 7.1 falls back to their
# Wikipedia lead image. Anything still without a photo after that appears on the globe
# with an imageless popup, and the "random site" button skips it.
no_photo = unique[unique["image_url"].isna()]
print(f"{len(no_photo)} site(s) with no photo in the export")
no_photo[["site_id", "label", "states", "component_count"]]

12 site(s) with no photo in the export


,site_id,label,states,component_count
379,1272,São Francisco Square in the Town of São Cristóvão,Brazil,1
653,955,Lorentz National Park,Indonesia,1
671,938,Sukur Cultural Landscape,Nigeria,1
797,783,Luther Memorials in Eisleben and Wittenberg,Germany,6
916,606,Serra da Capivara National Park,Brazil,1
978,335,Nanda Devi and Valley of Flowers National Parks,India,2
993,250,Great Living Chola Temples,India,3
1006,430,Frontiers of the Roman Empire,"United Kingdom of Great Britain and Northern Ireland,Germany",420
1008,452,Sundarbans National Park,India,1
1115,227,Comoé National Park,Côte d'Ivoire,1


---
## 4. Wikipedia article resolution (channel 2)

Wikidata stores the UNESCO site ID as property
[`P757`](https://www.wikidata.org/wiki/Property:P757), so a single batched SPARQL query
maps most of the list to a Wikidata item and its Wikipedia sitelinks — the same shape as
the IUCN `P627` lookup in the sister project. The query asks for sitelinks only; `P18`
images are deliberately not requested, since photos come from the export.

Sites the query misses (mostly properties inscribed at the latest session, not yet in
Wikidata) go through a name-based fallback chain:

1. **Wikidata entity search** (`wbsearchentities`) on each variant of the English name.
2. **Direct Wikipedia title lookup**, resolving redirects.

Name variants drop a trailing parenthesised country, a leading `The `, an em-dash
subtitle, and UNESCO title prefixes such as `Historic Centre of `.

In [15]:
site_ids = sorted(df["site_id"].dropna().astype(int).unique())
wikidata_map = ph.read_json_cache(WIKIDATA_CACHE)
print(f"{len(site_ids):,} site IDs to resolve · {len(wikidata_map):,} already cached")


1,273 site IDs to resolve · 1,119 already cached


In [16]:
if RUN_WIKIDATA_BATCH:
    pending = [i for i in site_ids if str(i) not in wikidata_map]
    print(f"querying Wikidata for {len(pending):,} sites")
    if pending:
        wikidata_map.update(ph.query_wikidata_batch(pending))
        ph.write_json_atomic(WIKIDATA_CACHE, wikidata_map)

resolved = {k for k, v in wikidata_map.items() if v.get("wiki_title")}
print(f"resolved by P757: {len(resolved):,} / {len(site_ids):,}")
if wikidata_map:
    # Which language editions the batch settled on. A long tail of small wikis is the
    # sign of a name match gone astray.
    print()
    print(pd.Series([v.get("wiki_language") for v in wikidata_map.values()])
            .value_counts().head(12).to_string())


querying Wikidata for 154 sites


Wikidata batches: 100%|██████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.31s/it]

resolved by P757: 1,119 / 1,273

en     906
de      75
fr      36
es      24
ja      18
nl      16
pt       9
ru       6
it       4
bg       3
gl       3
ceb      2


In [17]:
# Properties with no English article behind them, to investigate by hand. `no article`
# means nothing resolved at all; `<lang> only` means Wikidata lists that item's sitelinks
# without an English one. Pin a better article through MANUAL_ARTICLES below.
whc_no_english = ph.articles_without_english(df, wikidata_map)
print(whc_no_english["status"].value_counts().to_string())
print()
whc_no_english.head(25)


status
no article    154
de only        75
fr only        36
es only        24
ja only        18
nl only        16
pt only         9
ru only         6
it only         4
gl only         3
bg only         3
zh only         2
fi only         2
ceb only        2
hu only         2
sv only         2
he only         1
sh only         1
pl only         1
id only         1
et only         1
ca only         1
sr only         1
uk only         1
tr only         1



,site_id,label,states,status,wiki_title,wiki_lookup_source,wikidata_url
0,1758,The Bony Fish Fossils of the Western Limfjord – Evolution and Climate Adaptation in th...,Denmark,nl only,Fossielen_van_beenvissen_in_de_westelijke_Limfjord,wikidata_P757,https://www.wikidata.org/entity/Q127521592
1,1770,Alamūt Castle and Related Fortifications,Iran (Islamic Republic of),no article,NaN,NaN,NaN
2,1810,Mount Amel Castles,Lebanon,it only,Castelli_del_monte_Amel,wikidata_P757,https://www.wikidata.org/entity/Q140592613
3,1759,The Cemetery Complexes of the Xiongnu Nobility,Mongolia,nl only,Grafheuvels_van_de_aristocratie_van_de_Xiongnu,wikidata_P757,https://www.wikidata.org/entity/Q140697603
4,1715,Gdynia Modernist City Centre,Poland,no article,NaN,NaN,NaN
5,1766,Tashkent Modernist Architecture. Modernity and Tradition in Central Asia,Uzbekistan,fr only,Architecture_moderniste_de_Tachkent,wikidata_P757,https://www.wikidata.org/entity/Q124800237
6,1765,Jingdezhen Handicraft Porcelain Industry Sites,China,he only,אתרי_תעשיית_החרסינה_בעבודת_יד_של_ג'ינגדג'ן,wikidata_P757,https://www.wikidata.org/entity/Q140693109
7,1762,The system of Italian-style condominio theatres of the 18th and 19th centuries in Cent...,Italy,fr only,Théâtres_à_l'italienne_en_condominio,wikidata_P757,https://www.wikidata.org/entity/Q127466372
8,1774,Amazonia Theaters,Brazil,nl only,Amazoniatheaters,wikidata_P757,https://www.wikidata.org/entity/Q65561204
9,1768,The Medinas of the Historic Sultanates of the Comoros,Comoros,fr only,Médinas_des_Sultanats_historiques_des_Comores,wikidata_P757,https://www.wikidata.org/entity/Q65724639


In [18]:
# Name-based fallback for everything P757 did not resolve.
unresolved = [
    (int(r.site_id), r.label, r.states)
    for r in df.drop_duplicates("site_id").itertuples()
    if str(int(r.site_id)) not in resolved
]
print(f"{len(unresolved)} sites to resolve by name")
for site_id, label, _ in unresolved[:10]:
    print(f"  {site_id}  {label}")

154 sites to resolve by name
  1770  Alamūt Castle and Related Fortifications
  1715  Gdynia Modernist City Centre
  1719  The wider area of Mount Olympus
  1431  Coastal and Marine Ecosystems of the Bijagós Archipelago – Omatí Minhô
  13  Melka Kunture and Balchit: Archaeological and Palaeontological Sites in the Highland Area of Ethiopia
  1688  Cultural Landscape of Kenozero Lake
  1567  Funerary and memory sites of the First World War (Western Front)
  1686  Anticosti
  1608  Frontiers of the Roman Empire – The Danube Limes (Western Segment)
  1624  Chankillo Archaeoastronomical Complex


In [19]:
if RUN_NAME_FALLBACK and unresolved:
    name_map = ph.resolve_sites_by_name(unresolved, cache_path=NAME_CACHE)
    # Name-resolved entries never override a P757 hit.
    for key, entry in name_map.items():
        wikidata_map.setdefault(key, entry)
    ph.write_json_atomic(WIKIDATA_CACHE, wikidata_map)

still_missing = [
    int(r.site_id) for r in df.drop_duplicates("site_id").itertuples()
    if not (wikidata_map.get(str(int(r.site_id))) or {}).get("wiki_title")
]
print(f"still without a Wikipedia article: {len(still_missing)}")

Name fallback:  13%|████████▌                                                         | 20/154 [01:01<05:04,  2.27s/it]

  [sparql] HTTP 429 — waiting 120s


Name fallback:  38%|█████████████████████████▎                                        | 59/154 [04:51<03:02,  1.92s/it]

  [sparql] HTTP 502 — waiting 5s


Name fallback:  38%|█████████████████████████▎                                        | 59/154 [04:57<03:02,  1.92s/it]

  [sparql] HTTP 502 — waiting 10s


Name fallback:  49%|████████████████████████████████▏                                 | 75/154 [05:24<01:24,  1.07s/it]

  [sparql] HTTP 502 — waiting 5s


Name fallback: 100%|█████████████████████████████████████████████████████████████████| 154/154 [07:46<00:00,  3.03s/it]

  [name fallback] resolved 128 sites
still without a Wikipedia article: 26


In [ ]:
# Manual retry — pin a specific article onto a site the chain could not resolve.
# Fill the dict below, re-run, then re-run the attach cell.
MANUAL_ARTICLES = {
    # 1567: "Funerary and memory sites of the First World War (Western Front)",
}
for site_id, title in MANUAL_ARTICLES.items():
    entry = ph.wikipedia_direct_search(title, lang="en")
    if entry:
        wikidata_map[str(site_id)] = entry
        print(f"  {site_id} → {entry['wiki_url']}")
    else:
        print(f"  {site_id}: no Wikipedia page for {title!r}")
if MANUAL_ARTICLES:
    ph.write_json_atomic(WIKIDATA_CACHE, wikidata_map)

### 4.1 Cross-check against Wikipedia's own list

The chain above works one item at a time, so a site whose Wikidata entry carries no
`P757` or no English sitelink falls through it even when a perfectly good English
article exists.
[List of World Heritage Sites by year of inscription](https://en.wikipedia.org/wiki/List_of_World_Heritage_Sites_by_year_of_inscription)
catches those. Every row on it names both the article and the UNESCO record, so the
comparison joins on the identifier and never has to guess from a name.

Disagreements come back ranked by how actionable they are:

| status | what it means |
|---|---|
| `no_article` | the chain found nothing and the list names one |
| `other_language` | the chain found only a non-English edition |
| `different_article` | both name an English article and they differ. Often a redirect pair (*Via Appia* / *Appian Way*), so worth reading before adopting |
| `section_link` | the list points into a section of a broader article — *Ravenna* for the Early Christian Monuments of Ravenna — and the chain found nothing better. A fair place to send a reader, though the pageviews counted are the whole article's |
| `ours_is_narrower` | same, except the chain did find a dedicated English page: *Persian Qanat* against the list's *Qanat*. Ours is the more precise of the two, so it stands |

The cell after next adopts whichever of them you name, and does nothing by default.


In [ ]:
# Every row names both the article and the UNESCO record, so this joins on the site id.
whc_listed = {}
if RUN_ARTICLE_CROSSCHECK:
    whc_listed = ph.parse_whc_list(ph.fetch_wikitext(ph.WHC_LIST_PAGE))
    sections = sum(1 for entry in whc_listed.values() if entry["section"])
    print(f"{len(whc_listed):,} sites listed, {sections} of them via a section link")

whc_conflicts = ph.crosscheck_articles(df, wikidata_map, whc_listed)
print(whc_conflicts["status"].value_counts().to_string()
      if len(whc_conflicts) else "nothing to reconcile")
whc_conflicts.head(30)


In [ ]:
# Optional: adopt what the list names. Whole statuses, or specific site ids. Each title
# is resolved through Wikipedia so a redirect lands on its target. Nothing is adopted
# unless named here, section_link included.
APPLY_LISTED     = []   # e.g. ["no_article", "other_language"]
APPLY_LISTED_IDS = []   # e.g. [1752, 1758]

changed = ph.apply_listed_articles(wikidata_map, whc_conflicts,
                                   statuses=APPLY_LISTED, ids=APPLY_LISTED_IDS)
if changed:
    ph.write_json_atomic(WIKIDATA_CACHE, wikidata_map)
print(f"{len(changed)} article(s) replaced")


In [21]:
# Two sites resolving to the same article would share a popularity score. Keep it
# visible rather than silently deduplicating — some are genuine (transboundary
# inscriptions), some are a bad name match worth fixing above.
by_url = {}
for key, entry in wikidata_map.items():
    if entry.get("wiki_url"):
        by_url.setdefault(entry["wiki_url"], []).append(key)
shared = {url: keys for url, keys in by_url.items() if len(keys) > 1}
print(f"{len(shared)} Wikipedia articles shared by more than one site")

unique_sites = df.drop_duplicates("site_id")
names = pd.Series(unique_sites["label"].values, index=unique_sites["site_id"].astype(str))
for url, keys in list(shared.items())[:15]:
    print(f"  {url}")
    for key in keys:
        print(f"      {key}  {names.get(key, '?')}")


1 Wikipedia articles shared by more than one site
  https://en.wikipedia.org/wiki/Lower_Germanic_Limes
      1631  Frontiers of the Roman Empire – The Lower German Limes
      1608  Frontiers of the Roman Empire – The Danube Limes (Western Segment)


In [22]:
df = ph.attach_wikidata_fields(df, wikidata_map)
print(df[["label", "wiki_title", "wiki_language", "wiki_lookup_source"]].head(10).to_string())
print()
print("with article:", int(df.drop_duplicates("site_id")["wiki_title"].notna().sum()),
      "/", df["site_id"].nunique())

                                                                                                     label                                          wiki_title wiki_language      wiki_lookup_source
0  The Bony Fish Fossils of the Western Limfjord – Evolution and Climate Adaptation in the Earliest Eocene  Fossielen_van_beenvissen_in_de_westelijke_Limfjord            nl           wikidata_P757
1                                                                                              Aalto Works                                         Aalto_Works            en           wikidata_P757
2                                                            Beaches of the D-Day Landings, Normandy, 1944                                       D-Day_beaches            en           wikidata_P757
3                                                                         Ancient Buddhist Site of Sarnath                                             Sarnath            en           wikidata_P757
4              

### Checkpoint 2 — after Wikidata

In [23]:
CKPT2 = CKPT_DIR / "ckpt_2_wikidata.csv"
# Resume a later run from here by uncommenting: df = pd.read_csv(CKPT2)
df.to_csv(CKPT2, index=False)
print("saved", CKPT2, df.shape)

saved data/cache/checkpoints/ckpt_2_wikidata.csv (1273, 38)


---
## 5. Popularity — Wikipedia pageviews (channel 3)

Total human pageviews over the 12-month window, per resolved article. The `user` agent
filter excludes bots and crawlers.

Queried once per unique article, then filled back onto every label point of the site, so
a serial property with two points costs one request, not two.

`MIN_POPULARITY = 1` is a floor on the count: the globe sorts labels by `−popularity`,
so a 0 would suppress a label entirely rather than draw it at the smallest size.

In [24]:
articles = (
    df.dropna(subset=["wiki_title"])
      .drop_duplicates(subset=["wiki_project", "wiki_title"])[["wiki_project", "wiki_title"]]
)
print(f"{len(articles):,} unique articles to query (~{len(articles) * ph.SLEEP_PAGEVIEWS / 60:.0f} min)")

1,246 unique articles to query (~21 min)


In [ ]:
PAGEVIEWS_CACHE = CACHE_DIR / "pageviews" / f"pageviews_{PAGEVIEW_START}_{PAGEVIEW_END}.json"
pageviews = ph.read_json_cache(PAGEVIEWS_CACHE)
print(f"{len(pageviews):,} cached counts")

if RUN_PAGEVIEWS:
    ph.fetch_pageviews(articles.itertuples(index=False, name=None), pageviews, PAGEVIEWS_CACHE)
    print(f"cached {len(pageviews):,} counts → {PAGEVIEWS_CACHE}")


0 cached counts
querying 1,246 articles


Pageviews:  31%|█████████████████████                                               | 387/1246 [07:48<17:13,  1.20s/it]

In [ ]:
keys = df["wiki_project"].fillna("") + "|" + df["wiki_title"].fillna("")
df["popularity"] = keys.map(pageviews).fillna(0).astype(int).clip(lower=MIN_POPULARITY)

floor = int((df.drop_duplicates("site_id")["popularity"] == MIN_POPULARITY).sum())
print(f"sites at the {MIN_POPULARITY}-view floor: {floor}")
print()
top = df.drop_duplicates("site_id").nlargest(15, "popularity")
print(top[["label", "wiki_title", "popularity"]].to_string(index=False))

In [ ]:
# Sites on the floor despite having an article — usually a redirect title or a
# stub in a small-language edition. Worth a manual article override above.
suspect = df.drop_duplicates("site_id")
suspect = suspect[(suspect["popularity"] <= MIN_POPULARITY) & suspect["wiki_title"].notna()]
print(f"{len(suspect)} sites with an article but no views")
suspect[["site_id", "label", "wiki_project", "wiki_title"]].head(20)

### Checkpoint 3 — after pageviews

In [ ]:
CKPT3 = CKPT_DIR / "ckpt_3_pageviews.csv"
# Resume a later run from here by uncommenting: df = pd.read_csv(CKPT3)
df.to_csv(CKPT3, index=False)
print("saved", CKPT3, df.shape)

---
## 6. UNESCO Global Geoparks

A second UNESCO programme, a second export, and the same three questions: where is each
one, what does it look like, and how well known is it.

The geopark export is far simpler than the World Heritage one. Every geopark has a
published coordinate, so there is no component clustering and no manual placement. It
ships one photo per geopark and no photo metadata, one English introduction, and two
outbound links — a video and an official website — that the World Heritage export has no
equivalent of.

Geoparks are **not** a World Heritage category. They keep their own `category` and color
key, they are never "in danger" (that list is a World Heritage instrument), and they land
in the same five programme regions so a single region filter serves both datasets.

In [ ]:
graw = pd.read_csv(GEOPARK_CSV, encoding="utf-8-sig")

GEOPARK_EXPECTED = [
    "Internal ID", "Titre EN", "Pays", "Date", "Introduction EN", "Area Unit",
    "Area Total", "Population", "Transnational", "Main Image", "Video",
    "Site Internet", "Coordonnées", "URL",
]
missing = [c for c in GEOPARK_EXPECTED if c not in graw.columns]
print(f"{len(graw):,} rows × {graw.shape[1]} columns")
print("missing columns:", missing or "none")
print("area units:", set(graw["Area Unit"]))
print("duplicate internal IDs:", int(graw["Internal ID"].duplicated().sum()))
graw[["Internal ID", "Titre EN", "Pays", "Date", "Coordonnées"]].head()

In [ ]:
# One row per geopark. `Internal ID` (EUFR10, ASCN01) is the key: it is stable, unique,
# and it carries the programme region in its first two letters.
geopark_points = graw["Coordonnées"].map(ph.parse_coordinates)

gdf = pd.DataFrame({
    "dataset":           "geopark",
    "site_id":           graw["Internal ID"].map(ph.clean_str),
    "label":             graw["Titre EN"].map(ph.strip_tags),
    "short_description": graw["Introduction EN"].map(ph.description_text),
    "category":          ph.GEOPARK_CATEGORY,
    "color_key":         ph.GEOPARK_CATEGORY,
    "in_danger":         False,
    "region":            graw["Internal ID"].map(ph.geopark_region),
    "iso_codes":         graw["Pays"].map(lambda v: ph.clean_str(v).upper()),
    "states":            graw["Pays"].map(ph.country_names),
    "date_inscribed":    pd.to_datetime(graw["Date"], errors="coerce").dt.year.astype("Int64"),
    "area_hectares":     pd.to_numeric(graw["Area Total"], errors="coerce"),
    "transboundary":     graw["Transnational"].map(to_bool),
    "unesco_url":        graw["URL"].map(ph.clean_str),
    "video_url":         graw["Video"].map(ph.clean_str),
    "website_url":       graw["Site Internet"].map(ph.clean_str),
    "lat":               [point[0] for point in geopark_points],
    "lon":               [point[1] for point in geopark_points],
})

# The globe label drops the programme suffix every name ends in; the popup title and the
# search box keep the full `Titre EN`.
gdf["short_label"] = gdf["label"].map(lambda n: ph.geopark_short_label(n, SHORT_LABEL_MAX_CHARS))
gdf["short_description"] = gdf["short_description"].map(
    lambda t: ph.truncate_text(t, DESCRIPTION_MAX_CHARS))

# No clustering to do: one published coordinate, one label point, each its own source.
gdf["point_source"] = "geopark_coordinates"
gdf["label_rank"] = 1
gdf["label_count"] = 1

unplaced = gdf[gdf["lat"].isna() | gdf["lon"].isna()]
print(f"{len(gdf):,} geoparks, {len(unplaced)} without a usable coordinate")
if len(unplaced):
    print(unplaced[["site_id", "label"]].to_string(index=False))
print()
print(gdf["region"].value_counts().to_string())
print()
print("regions not in the index.html filter:", set(gdf["region"].dropna()) - set(ph.UNESCO_REGIONS) or "none")
print("unmapped country codes:", {c for codes in gdf["iso_codes"] for c in codes.split(",")
                                  if c and c not in ph.COUNTRY_NAMES} or "none")

In [ ]:
# Photos — straight from the export, exactly like section 3, and this takes every photo
# it holds: `Main Image` is the export's only image column, one URL per cell, with no
# gallery column and no URL anywhere in the text fields. So a geopark gets at most one
# photo, with no author, copyright or caption, hence no credit line and no caption
# overlay in the popup. The 44 with no photo fall back to Wikipedia in section 7.1.
def geopark_image(value):
    url = ph.clean_str(value)
    return url if url.lower().startswith(("http://", "https://")) else None


gdf["image_url"]        = graw["Main Image"].map(geopark_image)
gdf["extra_image_urls"] = ""
gdf["image_count"]      = gdf["image_url"].notna().astype(int)
gdf["image_source"]     = gdf["image_url"].map(lambda u: "UNESCO" if u else None)
for column in ("image_author", "image_copyright", "image_caption"):
    gdf[column] = None

print("geoparks with a photo  :", int(gdf["image_url"].notna().sum()), "/", len(gdf))
print("geoparks with a video  :", int((gdf["video_url"] != "").sum()))
print("geoparks with a website:", int((gdf["website_url"] != "").sum()))
print("with an introduction   :", int((gdf["short_description"] != "").sum()))
print()
print("introduction length:")
print(gdf["short_description"].str.len().describe().round(0).to_string())

### 6.1 Wikipedia article resolution

There is no geopark equivalent of `P757`. Wikidata has no UNESCO Global Geopark
identifier property at all, and the one historical identifier it carries —
[`P2467`](https://www.wikidata.org/wiki/Property:P2467), Global Geoparks Network ID
(former scheme) — sits on roughly 130 items, is formatted as `Portugal/6444`, and also
covers national geoparks that were never UNESCO-designated. It cannot be joined onto
`Internal ID`.

**So the field to match on is the name.** The first pass still goes through Wikidata
rather than straight to a text search: it pulls every item marked a UNESCO Global
Geopark — by designation ([`P1435`](https://www.wikidata.org/wiki/Property:P1435)), by
class (`P31`), or by that former GGN identifier — together with all of its labels and
aliases in eight languages, and matches our names against that closed set. Names are
compared with the programme words removed, which is what lets *Terres d'Hérault UNESCO
Global Geopark* meet *Géoparc mondial UNESCO des Terres d'Hérault*. A name two different
items answer to is dropped rather than guessed at.

About six in ten resolve that way. The rest fall through to the same name-based chain the
World Heritage list uses — Wikidata entity search, then a direct Wikipedia title lookup —
searched on the name without its programme suffix.

In [ ]:
geopark_map = ph.read_json_cache(GEOPARK_WIKIDATA_CACHE)
print(f"{len(gdf):,} geoparks to resolve · {len(geopark_map):,} already cached")

if RUN_GEOPARK_DESIGNATION:
    pending = [(sid, name) for sid, name in zip(gdf["site_id"], gdf["label"])
               if str(sid) not in geopark_map]
    print(f"resolving {len(pending):,} geoparks against the designated items")
    if pending:
        geopark_map.update(ph.resolve_geoparks_by_designation(pending))
        ph.write_json_atomic(GEOPARK_WIKIDATA_CACHE, geopark_map)

geopark_resolved = {k for k, v in geopark_map.items() if v.get("wiki_title")}
print(f"resolved by designation: {len(geopark_resolved):,} / {len(gdf):,}")


In [ ]:
# Name-based fallback for the rest, searched on the name without its programme suffix —
# "Psiloritis", not "Psiloritis UNESCO Global Geopark".
geopark_unresolved = [
    (str(row.site_id), ph.geopark_plain_name(row.label), row.states)
    for row in gdf.itertuples()
    if str(row.site_id) not in geopark_resolved
]
print(f"{len(geopark_unresolved)} geoparks to resolve by name")
for site_id, name, states in geopark_unresolved[:10]:
    print(f"  {site_id}  {name}  ({states})")

In [ ]:
if RUN_GEOPARK_NAME_FALLBACK and geopark_unresolved:
    geopark_names = ph.resolve_sites_by_name(geopark_unresolved, cache_path=GEOPARK_NAME_CACHE)
    # Name-resolved entries never override a designation hit.
    for key, entry in geopark_names.items():
        geopark_map.setdefault(key, entry)
    ph.write_json_atomic(GEOPARK_WIKIDATA_CACHE, geopark_map)

geopark_missing = [
    str(row.site_id) for row in gdf.itertuples()
    if not (geopark_map.get(str(row.site_id)) or {}).get("wiki_title")
]
print(f"still without a Wikipedia article: {len(geopark_missing)}")

### 6.2 Cross-check against Wikipedia's geopark lists

The same idea as section 4.1, over
[UNESCO Global Geoparks](https://en.wikipedia.org/wiki/UNESCO_Global_Geoparks) and the
five regional lists it links to. These pages carry no identifier, so the join is on the
plain name — *Fangshan*, not *Fangshan UNESCO Global Geopark* — matched the same way the
designation index is. They also cover fewer geoparks than the export does, so a geopark
absent from them means nothing at all.


In [ ]:
# No identifiers on these pages, so the join is on the plain name. Both halves of each
# link are indexed, since the table writes [[Fangshan District|Fangshan]].
geopark_listed = {}
if RUN_ARTICLE_CROSSCHECK:
    for page in ph.GEOPARK_LIST_PAGES:
        ph.parse_geopark_list(ph.fetch_wikitext(page), geopark_listed)
    print(f"{len(geopark_listed):,} names indexed from {len(ph.GEOPARK_LIST_PAGES)} pages")

geopark_conflicts = ph.crosscheck_articles(
    gdf, geopark_map, geopark_listed,
    key=lambda row: ph.geopark_name_key(ph.geopark_plain_name(row["label"])))
print(geopark_conflicts["status"].value_counts().to_string()
      if len(geopark_conflicts) else "nothing to reconcile")
geopark_conflicts.head(30)


In [ ]:
# Optional: adopt what the lists name. Whole statuses, or specific Internal IDs.
# Nothing is adopted unless named here, section_link included.
APPLY_GEOPARK_LISTED     = []   # e.g. ["no_article"]
APPLY_GEOPARK_LISTED_IDS = []   # e.g. ["ASCN01", "ASID01"]

changed = ph.apply_listed_articles(geopark_map, geopark_conflicts,
                                   statuses=APPLY_GEOPARK_LISTED,
                                   ids=APPLY_GEOPARK_LISTED_IDS)
if changed:
    ph.write_json_atomic(GEOPARK_WIKIDATA_CACHE, geopark_map)
print(f"{len(changed)} geopark article(s) replaced")


In [ ]:
# Manual retry — pin a specific article onto a geopark the chain could not resolve, or
# correct a bad match. Same shape as MANUAL_ARTICLES in section 4, keyed by Internal ID.
GEOPARK_ARTICLE_OVERRIDES = {
    # "EUFR10": "Hérault",
}
for geopark_id, title in GEOPARK_ARTICLE_OVERRIDES.items():
    entry = ph.wikipedia_direct_search(title, lang="en")
    if entry:
        geopark_map[geopark_id] = entry
        print(f"  {geopark_id} → {entry['wiki_url']}")
    else:
        print(f"  {geopark_id}: no Wikipedia page for {title!r}")
if GEOPARK_ARTICLE_OVERRIDES:
    ph.write_json_atomic(GEOPARK_WIKIDATA_CACHE, geopark_map)

gdf = ph.attach_wikidata_fields(gdf, geopark_map)
print("with article:", int(gdf["wiki_title"].notna().sum()), "/", len(gdf))
print()
print(gdf["wiki_lookup_source"].value_counts(dropna=False).to_string())


In [ ]:
# Same check over the geoparks. A geopark is far likelier than a World Heritage property
# to have only a local-language article, so `<lang> only` here is usually genuine rather
# than a bad match — override through GEOPARK_ARTICLE_OVERRIDES above when it is not.
geopark_no_english = ph.articles_without_english(gdf, geopark_map)
print(geopark_no_english["status"].value_counts().to_string())
print()
geopark_no_english.head(25)


### 6.3 Popularity

Same window, same endpoint and the same shared cache as section 5, so an article a
geopark happens to share with a World Heritage property is only ever fetched once.

In [ ]:
geopark_articles = (
    gdf.dropna(subset=["wiki_title"])
       .drop_duplicates(subset=["wiki_project", "wiki_title"])[["wiki_project", "wiki_title"]]
)
print(f"{len(geopark_articles):,} unique articles "
      f"(~{len(geopark_articles) * ph.SLEEP_PAGEVIEWS / 60:.0f} min)")

if RUN_GEOPARK_PAGEVIEWS:
    ph.fetch_pageviews(geopark_articles.itertuples(index=False, name=None), pageviews,
                       PAGEVIEWS_CACHE, desc="Geopark pageviews")

geopark_keys = gdf["wiki_project"].fillna("") + "|" + gdf["wiki_title"].fillna("")
gdf["popularity"] = geopark_keys.map(pageviews).fillna(0).astype(int).clip(lower=MIN_POPULARITY)

print(f"geoparks at the {MIN_POPULARITY}-view floor:",
      int((gdf["popularity"] == MIN_POPULARITY).sum()))
print()
print(gdf.nlargest(15, "popularity")[["label", "wiki_title", "popularity"]].to_string(index=False))


### Checkpoint 4 — after the geoparks

In [ ]:
CKPT4 = CKPT_DIR / "ckpt_4_geoparks.csv"
# Resume a later run from here by uncommenting: gdf = pd.read_csv(CKPT4)
gdf.to_csv(CKPT4, index=False)
print("saved", CKPT4, gdf.shape)

### 6.4 One globe

The two datasets share a schema from here on: the same fields, the same five regions,
one `dataset` column saying which export a point came from. Coordinates are de-duplicated
across both, so a geopark that sits on the same point as a World Heritage property is
nudged apart rather than hidden behind it.

In [ ]:
all_sites = pd.concat([df, gdf], ignore_index=True, sort=False)
if JITTER_DUPLICATE_POINTS:
    all_sites = ph.jitter_duplicate_points(all_sites)

duplicates = all_sites.groupby([all_sites["lat"].round(6), all_sites["lon"].round(6)]).size()
print(all_sites["dataset"].value_counts().to_string())
print()
print(f"{all_sites['site_id'].nunique():,} sites → {len(all_sites):,} label points")
print("coordinate pairs still shared by 2+ label points:", int((duplicates > 1).sum()))
print()
print(pd.crosstab(all_sites.drop_duplicates("site_id")["region"],
                  all_sites.drop_duplicates("site_id")["color_key"],
                  margins=True).to_string())

---
## 7. Photos

Two things happen here: 7.1 finds a photo for the sites whose export has none, and 7.2
gets every photo to the browser at the size the card actually shows.

### 7.1 Wikipedia fallback

12 World Heritage properties and 44 geoparks have an empty image column. Those fall back
to the lead image of their Wikipedia article, via the `pageimages` API at the same 760 px
the popup asks for.

A second call reads the Commons file's description page for the credit: `Artist` and
`LicenseShortName` become the popup's credit line, and `ObjectName` its caption when that
is short enough to read as one. A site whose article has no lead image, or that has no
article yet, keeps an imageless popup.

`image_source` records which source a photo came from, `UNESCO` or `Wikipedia`.


In [ ]:
# The sites with no photo of their own, where there is an article to fall back to.
unique_all = all_sites.drop_duplicates("site_id")
photoless = unique_all[unique_all["image_url"].isna()]
borrowable = photoless[photoless["wiki_title"].notna()]

print(f"{len(photoless)} site(s) with no photo in either export")
print(f"  {len(borrowable)} with a Wikipedia article to borrow from")
print(f"  {len(photoless) - len(borrowable)} with no article either — these stay imageless")

if RUN_WIKIPEDIA_THUMBNAILS and len(borrowable):
    thumbnails = ph.fetch_wikipedia_thumbnails(
        zip(borrowable["site_id"], borrowable["wiki_project"], borrowable["wiki_title"]),
        cache_path=THUMBNAIL_CACHE,
    )
else:
    thumbnails = ph.read_json_cache(THUMBNAIL_CACHE)

found = {key: record for key, record in thumbnails.items() if record.get("image_url")}
print(f"\n{len(found)} lead photo(s) available"
      f" ({len(thumbnails) - len(found)} article(s) checked and had none)")


In [ ]:
# Write them onto the frame, leaving rows that already have a photo untouched.
keys = all_sites["site_id"].astype(str)
gap = all_sites["image_url"].isna() & keys.isin(found)

for column in ("image_url", "image_author", "image_copyright", "image_caption", "image_source"):
    all_sites.loc[gap, column] = keys[gap].map(lambda k: found[k].get(column))
all_sites.loc[gap, "image_count"] = 1

unique_all = all_sites.drop_duplicates("site_id")
print(unique_all["image_source"].value_counts(dropna=False).to_string())
print(f"\nsites still without a photo: {int(unique_all['image_url'].isna().sum())}")
print()
borrowed = all_sites[gap].drop_duplicates("site_id")
print(borrowed[["dataset", "site_id", "label", "image_author", "image_copyright"]]
      .to_string(index=False))


### 7.2 Delivery — the resizing proxy

UNESCO publishes photographs, not web assets. The 197 geopark main photos come to **362 MB
between them** — mean 1.9 MB, 37 over 1 MB, 17 over 8 MB, the largest a 41 MB stereo JPEG
(`image/mpo`) of a Saudi crater. Neither host offers a resized variant: Azure Blob Storage
ignores `?width=`, and the World Heritage Centre serves each file as it was uploaded.

So `index.html` requests every photo through a **resizing proxy** ([wsrv.nl](https://wsrv.nl))
at the 760 px the card actually shows. Measured over all 197 geopark photos: **362 MB → 12 MB**,
mean 64 kB, largest 172 kB. A sample of 40 World Heritage main photos goes through the same
way at a mean of 80 kB — worth noting, because the proxy fetches `whc.unesco.org` successfully
where a scripted request gets a 403.

That rewrite lives in the browser rather than in `sites.geojson`, for two reasons: it then
covers the 25,497 gallery URLs too without adding ~1.3 MB of rewritten links to the file
every visitor downloads, and each `<img>` can fall back to UNESCO's own URL by itself if the
proxy ever fails.

**What the proxy cannot do is ingest an image over 71 megapixels**, and seven photos exceed
that — two geoparks and five World Heritage properties, the largest 128 MP. This section
finds them and shrinks them here instead — the only images the pipeline ever downloads.
Results go to `images/`, which is committed, and their `image_url` points at the local
file; the browser leaves relative paths unproxied.

Only *main* photos are checked. Checking all 25,497 gallery URLs would take around half an
hour, and an oversized gallery slide simply falls back to its original: slow for that one
slide, never broken.

In [ ]:
# One HEAD per distinct main photo, through the proxy, to find what it refuses.
# Results are cached — the pixel limit is a property of the file, not a transient error.
main_photos = sorted(all_sites["image_url"].dropna().unique())
print(f"{len(main_photos):,} distinct main photos")

proxy_checks = {}
if RUN_PROXY_CHECK:
    proxy_checks = ph.check_proxy_images(main_photos, cache_path=PROXY_CACHE)
else:
    proxy_checks = ph.read_json_cache(PROXY_CACHE)

checked = [proxy_checks[u] for u in main_photos if u in proxy_checks]
refused = sorted(u for u in main_photos if u in proxy_checks and not proxy_checks[u]["ok"])
served = [c["bytes"] for c in checked if c["ok"] and c["bytes"]]

print(f"checked {len(checked):,}  ·  proxy serves {len(checked) - len(refused):,}  ·  refuses {len(refused)}")
if served:
    print(f"proxied size: mean {sum(served) / len(served) / 1024:.0f} kB, "
          f"max {max(served) / 1024:.0f} kB, total {sum(served) / 1048576:.1f} MB")
for url in refused:
    print(f"  {proxy_checks[url]['status']} {proxy_checks[url]['message']}  {url.rsplit('/', 1)[-1][:56]}")

In [ ]:
# Shrink whatever the proxy refused, and point those rows at the local file. An already
# reduced file is left alone, so re-running costs nothing.
reduced = {}
for url in refused:
    rows = all_sites[all_sites["image_url"] == url]
    if rows.empty:
        continue
    row = rows.iloc[0]
    folder = IMAGE_DIR / ("geoparks" if row["dataset"] == "geopark" else "whc")
    destination = folder / f"{row['site_id']}.jpg"
    # A file reduced by an earlier run is used whatever the flag says: RUN_LOCAL_REDUCTION
    # gates the download, not the result of one. Otherwise turning the flag off would quietly
    # point these rows back at their 40 MB originals.
    if destination.exists():
        reduced[url] = destination.as_posix()
    elif RUN_LOCAL_REDUCTION and ph.reduce_image_locally(url, destination):
        reduced[url] = destination.as_posix()

if reduced:
    all_sites["image_url"] = all_sites["image_url"].map(lambda u: reduced.get(u, u))

still_refused = [u for u in refused if u not in reduced]
print()
print(f"{len(reduced)} photo(s) now served from {IMAGE_DIR}/")
print(f"{len(still_refused)} still on the original URL "
      f"(the popup falls back to it, at full size)")
for url in still_refused:
    print(f"  {url}")
print()
print(all_sites.loc[all_sites["image_url"].isin(reduced.values()),
                    ["dataset", "site_id", "label", "image_url"]].to_string(index=False))

---
## 8. Export

`sites.geojson` carries only the fields `index.html` reads. One feature per label point,
so a property with two label points appears twice, distinguished by `label_rank`.

In [ ]:
geojson = ph.build_geojson(all_sites)
ph.write_geojson(geojson, OUTPUT_PATH)
print(f"  World Heritage export {WHC_EXPORT_DATE} · geoparks export {GEOPARK_EXPORT_DATE}")

example = max(geojson["features"], key=lambda f: f["properties"].get("popularity", 0))
print()
print(json.dumps(example, ensure_ascii=False, indent=2)[:1800])

In [ ]:
# Field coverage in the exported file
coverage = pd.Series({
    field: sum(1 for f in geojson["features"] if field in f["properties"])
    for field in ph.GEOJSON_FIELDS
})
total = len(geojson["features"])
pd.DataFrame({"present": coverage, "share": (coverage / total).round(3)})

---
## 9. Quality checks

Run these after an export to see what the globe will actually look like.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Log-log, because the distribution is a power law: a few sites carry millions of views
# and the tail sits in the hundreds. On linear axes that tail is a single black bar.
popularity = all_sites.drop_duplicates("site_id")["popularity"]
bins = np.logspace(0, np.log10(max(int(popularity.max()), 10)), 60)

plt.figure(figsize=(6, 3.4))
plt.hist(popularity, bins=bins, color="#A538F0")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("views / year")
plt.title("Pageviews per site")
plt.tight_layout()
plt.show()

print(popularity.describe().round(0).to_string())


In [ ]:
# Placement and photo coverage, end to end. Coverage by region and colour is not
# repeated here: section 6.4 already prints that crosstab.
unique = all_sites.drop_duplicates("site_id")
print(unique["point_source"].value_counts().to_string())
print()
print("sites exported:", unique["site_id"].nunique())
print("with a photo  :", int(unique["image_url"].notna().sum()))
print("with an article:", int(unique["wiki_title"].notna().sum()))

In [ ]:
# Names still long enough to crowd the globe at low zoom
long_labels = unique.assign(display=unique["short_label"].fillna(unique["label"]))
long_labels = long_labels[long_labels["display"].str.len() > SHORT_LABEL_MAX_CHARS]
print(f"{len(long_labels)} labels longer than {SHORT_LABEL_MAX_CHARS} characters")
long_labels[["site_id", "label", "short_label"]].head(20)

In [ ]:
# Manual label override — set a shorter on-globe name for specific sites, then re-export.
LABEL_OVERRIDES = {
    # 86: "Pyramids of Giza",
}
for site_id, text in LABEL_OVERRIDES.items():
    all_sites.loc[all_sites["site_id"] == site_id, "short_label"] = text
if LABEL_OVERRIDES:
    ph.write_geojson(ph.build_geojson(all_sites), OUTPUT_PATH)